# Bài toán Visual Question Answering (VQA) - Baseline

Notebook này triển khai một mô hình baseline cho bài toán VQA sử dụng ResNet-18 để trích xuất đặc trưng hình ảnh và LSTM để xử lý câu hỏi.

# Thư viện và Framework

In [1]:
import os
import json
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import collections
import csv
from tqdm import tqdm
import torch.backends.cudnn as cudnn
import matplotlib.pyplot as plt
import random
import numpy as np
from sklearn.metrics import f1_score

# Set experiment

In [2]:
SEED = 42

def set_seed(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    # Optional: deterministic mạnh hơn
    # torch.use_deterministic_algorithms(True)


set_seed(SEED)
g = torch.Generator()
g.manual_seed(SEED)

# Config


In [ ]:
class Path:
    # Đường dẫn gốc của dữ liệu
    BASE_DIR = "/kaggle/input/datasets/lbophc/nckh-vqa"

    # Đường dẫn file JSON câu hỏi
    TRAIN_SAMPLES_JSON = "/kaggle/input/datasets/lbophc/samples/train_samples.json"
    VAL_SAMPLES_JSON = "/kaggle/input/datasets/lbophc/samples/val_samples.json"
    ORIGINAL_JSON = os.path.join(BASE_DIR, "questions/original.json")
    TRAIN_JSON = os.path.join(BASE_DIR, "questions/train.json")
    VAL_JSON = os.path.join(BASE_DIR, "questions/val.json")
    TEST_JSON  = os.path.join(BASE_DIR, "questions/test.json")

    # Thư mục chứa hình ảnh
    TRAIN_IMG_DIR = os.path.join(BASE_DIR, "images/train")
    TEST_IMG_DIR  = os.path.join(BASE_DIR, "images/test")

    # Đường dẫn lưu trữ model và vocab
    SAVE_PATH = "/kaggle/working/best_model.pth"
    HISTORY_JSON_PATH = "/kaggle/working/training_history.json"
    HISTORY_PLOT_PATH = "/kaggle/working/training_history.png"
    VOCAB_PATH = "/kaggle/input/models/lbophc/vocab/pytorch/default/1/vocab.pth"
    ANS_PATH = "/kaggle/input/models/lbophc/vocab/pytorch/default/1/ans_vocab.pth"
    SUBMISSION_PATH = "/kaggle/working/submission.csv"

class Config:
    # Tham số huấn luyện
    BATCH_SIZE = 64
    EPOCHS = 12
    LR = 1e-4
    MAX_LEN = 25 # Độ dài tối đa của câu hỏi
    VOCAB = torch.load(Path.VOCAB_PATH) if os.path.exists(Path.VOCAB_PATH) else None
    ANS_VOCAB = torch.load(Path.ANS_PATH) if os.path.exists(Path.ANS_PATH) else None


## 📊 Dataset & DataLoader
Định nghĩa lớp `VQADataset` để tải dữ liệu, xử lý câu hỏi thành token và chuẩn bị hình ảnh.

In [4]:
class VQADataset(Dataset):
    def __init__(self, path, img_dir, vocab=None, ans_vocab=None, train=True, val=True, test=False):
        with open(path, "r") as f:
            if test:
                self.data = json.load(f)["questions"]
            else: 
                self.data = json.load(f)["root"]["questions"]

        self.img_dir = img_dir
        self.train = train
        self.val = val
        self.vocab = vocab
        self.ans_vocab = ans_vocab

        # Tự động xây dựng từ điển nếu chưa có
        if vocab is None:
            self.vocab = self.build_vocab()

        if train and ans_vocab is None:
            self.ans_vocab = self.build_ans_vocab()

    def build_vocab(self):
        """Xây dựng từ điển cho câu hỏi"""
        words = []
        for q in self.data:
            words.extend(q["question"].lower().replace("?", "").replace(";", "").split())

        counter = collections.Counter(words)
        vocab = {w: i+2 for i, (w, _) in enumerate(counter.items())}
        vocab["<PAD>"] = 0
        vocab["<UNK>"] = 1
        return vocab

    def build_ans_vocab(self):
        """Xây dựng từ điển cho câu trả lời (nhãn)"""
        answers = [q["answer"] for q in self.data if "answer" in q]
        ans_vocab = {a: i for i, a in enumerate(sorted(set(answers)))}
        return ans_vocab

    def __len__(self):
        return len(self.data)

    def encode_q(self, q):
        """Chuyển câu hỏi văn bản thành tensor số"""
        q = q.lower().replace("?", "").replace(";", "").split()
        ids = [self.vocab.get(w, 1) for w in q]
        if len(ids) < Config.MAX_LEN:
            ids += [0] * (Config.MAX_LEN - len(ids))
        return torch.tensor(ids[:Config.MAX_LEN])

    def __getitem__(self, idx):
        q = self.data[idx]

        # Đọc ảnh
        img_path = os.path.join(self.img_dir, q["image_filename"])
        img = Image.open(img_path).convert("RGB")

        # Transform ảnh và encode câu hỏi
        img = transform(img)
        q_tokens = self.encode_q(q["question"])

        if (self.train and "answer" in q) or (self.val and "answer" in q):
            label = self.ans_vocab[q["answer"]]
            return img, q_tokens, label
        else:
            return img, q_tokens, q["qid"]

# =========================
# 🖼 TRANSFORM ẢNH
# =========================
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [5]:
class Data_Loader:
    def __init__(self):
        # vocab and vocab_ans
        self.vocab = Config.VOCAB
        self.ans_vocab = Config.ANS_VOCAB

    def load_data(self, path_json, img_dir, train=True, val=False, test=False):
        ds = VQADataset(path=path_json, img_dir=img_dir, 
                        vocab=self.vocab, ans_vocab=self.ans_vocab, 
                        train=train, val=val, test=test)
        return DataLoader(
            ds, 
            batch_size=Config.BATCH_SIZE, 
            shuffle=train, 
            num_workers=4,        # Increased workers
            prefetch_factor=2,     # Prefetch batches
            pin_memory=True,
            generator=g
        )

## 🧱 Kiến trúc Mô hình (Model)
Sử dụng ResNet-18 (pre-trained) cho phần nhìn và LSTM cho phần ngôn ngữ. Sau đó gộp hai đặc trưng lại để phân loại câu trả lời.

In [6]:
class Model_ResNet_LSTM(nn.Module):
    def __init__(self, vocab_size, num_answers):
        super().__init__()

        # CNN trích xuất đặc trưng ảnh (bỏ lớp fully connected cuối)
        resnet = models.resnet18(weights="IMAGENET1K_V1")
        self.cnn = nn.Sequential(*list(resnet.children())[:-2])

        # Nhánh xử lý câu hỏi (Embedding + LSTM)
        self.embedding = nn.Embedding(vocab_size, 128)
        self.lstm = nn.LSTM(128, 256, batch_first=True)

        # Bộ phân loại gộp đặc trưng ảnh và text
        self.classifier = nn.Sequential(
            nn.Linear(512 + 256, 512),
            nn.ReLU(),
            nn.Linear(512, num_answers)
        )

    def forward(self, img, q):
        # Trích xuất đặc trưng ảnh: (batch, 512, 7, 7) -> (batch, 512)
        v = self.cnn(img).mean([2, 3])

        # Trích xuất đặc trưng câu hỏi
        q = self.embedding(q)
        _, (h, _) = self.lstm(q)
        q = h.squeeze(0) # (batch, 256)

        # Gộp (Concatenate) và phân loại
        x = torch.cat([v, q], dim=1)
        return self.classifier(x)

In [7]:
class Model_ViT_Cross_attn(nn.Module):
    def __init__(self, vocab_size, num_answers, i_hidden=196, i_embed=768, q_embed=256):
        super().__init__()
        # transformers img
        self.vit = models.vit_b_16(weights="IMAGENET1K_V1")
        # embedding question
        self.embed = nn.Embedding(vocab_size, q_embed, padding_idx=0)
        # cls + position question
        self.q_cls_token = nn.Parameter(
            torch.randn(1, 1, q_embed) * 0.02
        )
        self.q_pos_embed = nn.Parameter(
            torch.randn(1, Config.MAX_LEN + 1, q_embed) * 0.02
        )
        # transformer for question 
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=q_embed,
            nhead=8,
            batch_first=True
        )
        self.trans = nn.TransformerEncoder(
            encoder_layer,
            num_layers=4
        )
        # layer normalization
        self.i_layerN = nn.LayerNorm(i_embed)
        self.q_layerN = nn.LayerNorm(q_embed)
        # Cross-attention
        self.linear = nn.Linear(i_embed, q_embed)
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=q_embed,
            num_heads=8,
            batch_first=True
        )
        # classifier
        self.classifier = nn.Sequential(
            nn.Linear(q_embed, 512),
            nn.ReLU(),
            nn.Linear(512, num_answers)
        )
        # ffn
        self.cross_n1 = nn.LayerNorm(q_embed)
        self.cross_n2 = nn.LayerNorm(q_embed)
        self.ffn = nn.Sequential(
            nn.Linear(q_embed, q_embed * 4),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(q_embed * 4, q_embed)
        )

    def get_vit_token(self, img):
        with torch.no_grad():
            i = self.vit._process_input(img)
            cls_token = self.vit.class_token.expand(
                i.shape[0], -1, -1
            )
            i = torch.cat([cls_token, i], dim=1)
            i = self.vit.encoder(i)
        return self.i_layerN(i)

    def forward(self, img, q):
        # trans img
        i = self.get_vit_token(img)
        i = self.linear(i)
        # processing question
        ## mask
        pad_mask = (q == 0)
        ## embed
        q = self.embed(q)
        ## cls
        batch_size = q.size(0)
        cls_token = self.q_cls_token.expand(batch_size, -1, -1)
        q = torch.cat([cls_token, q], dim=1)
        ## cls mask
        cls_mask = torch.zeros(batch_size, 1, dtype=torch.bool, device=q.device)
        pad_mask = torch.cat([cls_mask, pad_mask], dim=1)
        ## position
        q = q + self.q_pos_embed[:, :q.size(1), :]
        q = self.trans(q, src_key_padding_mask=pad_mask)
        q = self.q_layerN(q)
        # cross attention
        q_attn, _ = self.cross_attn(query=q, key=i, value=i)
        q = self.cross_n1(q + q_attn)
        q = self.cross_n2(q + self.ffn(q))
        q_out = q[:, 0, :]
        return self.classifier(q_out)

## 🎯 Huấn luyện (Training)
Hàm thực hiện quá trình training model.

In [9]:
class Tranning:
    def __init__(self, model, train_loader, val_loader):
        # Lưu loss theo từng epoch
        self.history = {
            "epoch": [],
            "train_loss": [],
            "val_loss": [],
            "train_acc": [],
            "val_acc": [],
            "train_f1": [],
            "val_f1": [],
        }

        # Lưu best validation loss
        self.best_val_loss = float("inf")
        self.best_val_acc = 0.0
        self.best_val_f1 = 0.0
        self.best_epoch = 0

        # model
        self.model = model
        # dataloader
        self.t_loader = train_loader
        self.v_loader = val_loader

        # optimizer
        self.opt = optim.Adam(
            filter(
                lambda m: m.requires_grad, 
                model.parameters()
            ), 
            lr=Config.LR
        )
        # loss
        self.loss_fn = nn.CrossEntropyLoss()
        # device
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        # Initialize AMP Scaler
        self.use_amp = self.device.type == "cuda"
        self.scaler = torch.amp.GradScaler('cuda', enabled=self.use_amp)
 
    def train(self):
        
        print(f"Đang sử dụng thiết bị: {self.device}")
    
        # Enable cuDNN benchmark for static input sizes
        cudnn.benchmark = True 
        self.model.to(self.device)
        # frozen ViT
        if hasattr(self.model, 'vit'):
            for param in self.model.vit.parameters():
                param.requires_grad = False

            for block in self.model.vit.encoder.layers[-4:]:
                for param in block.parameters():
                    param.requires_grad = True 
        
        for epoch in range(Config.EPOCHS):
            # train
            self.model.train()
            # if hasattr(self.model, 'vit'):
            #     self.model.vit.eval()
            total_train_loss = 0
            train_correct = 0
            train_samples = 0
            all_train_preds = []
            all_train_targets = []

            pbar = tqdm(self.t_loader, desc=f"Epoch {epoch+1:02d}/{Config.EPOCHS:02d} [Train]")
        
            for img, q, y in pbar:
                # pin_memory allows non_blocking transfers
                img = img.to(self.device, non_blocking=True)
                q = q.to(self.device, non_blocking=True)
                y = y.to(self.device, non_blocking=True)
        
                # 4. set_to_none=True is slightly faster than zeroing
                self.opt.zero_grad(set_to_none=True)
                
                # 5. Mixed Precision Context
                with torch.amp.autocast('cuda', enabled=self.use_amp):
                    pred = self.model(img, q)
                    loss = self.loss_fn(pred, y)
        
                # Scale loss and backpropagate
                self.scaler.scale(loss).backward()
                self.scaler.step(self.opt)
                self.scaler.update()

                total_train_loss += loss.item() * y.size(0)
                preds = pred.argmax(dim=1)
                train_correct += (preds == y).sum().item()
                train_samples += y.size(0)

                all_train_preds.extend(preds.detach().cpu().numpy())
                all_train_targets.extend(y.detach().cpu().numpy())
        
                pbar.set_postfix(
                    loss=f"{loss.item():.4f}", 
                    acc=f"{train_correct/train_samples:.4f}"
                )
                
            # Loss trung bình của epoch
            epoch_train_loss = total_train_loss / train_samples
            epoch_train_acc = train_correct / train_samples
            epoch_train_f1 = f1_score(all_train_targets, all_train_preds, average="weighted", zero_division=0)
            # val
            self.model.eval()
            total_val_loss = 0.0
            val_correct = 0
            val_samples = 0
            all_val_preds = []
            all_val_targets = []
            with torch.no_grad():
                for img, q, y in tqdm(self.v_loader, desc=f"Epoch {epoch+1:02d}/{Config.EPOCHS:02d} [Val]"):
                    # pin_memory allows non_blocking transfers
                    img = img.to(self.device, non_blocking=True)
                    q = q.to(self.device, non_blocking=True)
                    y = y.to(self.device, non_blocking=True)
                    
                    with torch.amp.autocast('cuda', enabled=self.use_amp):
                        pred = self.model(img, q)
                        loss = self.loss_fn(pred, y)

                    total_val_loss += loss.item() * y.size(0)
                    preds = pred.argmax(dim=1)
                    val_correct += (preds == y).sum().item()
                    val_samples += y.size(0)

                    all_val_preds.extend(preds.cpu().numpy())
                    all_val_targets.extend(y.cpu().numpy())

            epoch_val_loss = total_val_loss / val_samples
            epoch_val_acc = val_correct / val_samples
            epoch_val_f1 = f1_score(all_val_targets, all_val_preds, average="weighted", zero_division=0)
            self.best_val_f1 = max(epoch_val_f1, self.best_val_f1)
            # history
            self.history["epoch"].append(epoch + 1)
            self.history["train_loss"].append(epoch_train_loss)
            self.history["val_loss"].append(epoch_val_loss)
            self.history["train_acc"].append(epoch_train_acc)
            self.history["val_acc"].append(epoch_val_acc)
            self.history["train_f1"].append(epoch_train_f1)
            self.history["val_f1"].append(epoch_val_f1)
            
            if epoch_val_loss < self.best_val_loss:
                self.best_val_loss = epoch_val_loss
                self.best_val_acc = epoch_val_acc
                self.best_epoch = epoch + 1
                torch.save({
                    "epoch": epoch + 1,
                    "model_state_dict": self.model.state_dict(),
                    "optimizer_state_dict": self.opt.state_dict(),
                    "best_val_loss": self.best_val_loss,
                    "best_val_acc": self.best_val_acc,
                    "best_val_f1": self.best_val_f1,
                    "history": self.history
                }, Path.SAVE_PATH)
                best_text = " ★ [BEST MODEL SAVED]"
            else:
                best_text = ""
                    
            print(
                f"Epoch {epoch+1:02d}/{Config.EPOCHS:02d} | "
                f"Train: Loss {epoch_train_loss:.4f} - Acc {epoch_train_acc:.4f} - F1 {epoch_train_f1:.4f} | "
                f"Val: Loss {epoch_val_loss:.4f} - Acc {epoch_val_acc:.4f} - F1 {epoch_val_f1:.4f}"
                f"{best_text}"
            )


        # Lưu history
        with open(Path.HISTORY_JSON_PATH, "w", encoding="utf-8") as f:
            json.dump(self.history, f, indent=4)
        print(f"\n✔ Đã lưu lịch sử huấn luyện tại: {Path.HISTORY_JSON_PATH}")
        print(f"✔ Best Val Loss: {self.best_val_loss:.4f} | Best Val F1: {self.best_val_f1:.4f} (Epoch {self.best_epoch})")

    def plot_history(self, save_path=Path.HISTORY_PLOT_PATH):
        """Vẽ biểu đồ so sánh Loss, Accuracy và F1-Score"""
        epochs = self.history["epoch"]
        if not epochs:
            print("Chưa có dữ liệu history để vẽ.")
            return
        fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))
        # 1. Biểu đồ Loss
        ax1.plot(epochs, self.history["train_loss"], "o-", label="Train Loss", color="royalblue")
        ax1.plot(epochs, self.history["val_loss"], "s-", label="Val Loss", color="crimson")
        ax1.set_title("Loss", fontsize=13)
        ax1.set_xlabel("Epoch")
        ax1.set_ylabel("Loss")
        ax1.legend()
        ax1.grid(True, linestyle="--", alpha=0.6)
        # 2. Biểu đồ Accuracy
        ax2.plot(epochs, self.history["train_acc"], "o-", label="Train Acc", color="seagreen")
        ax2.plot(epochs, self.history["val_acc"], "s-", label="Val Acc", color="darkorange")
        ax2.set_title("Accuracy", fontsize=13)
        ax2.set_xlabel("Epoch")
        ax2.set_ylabel("Accuracy")
        ax2.legend()
        ax2.grid(True, linestyle="--", alpha=0.6)
        # 3. Biểu đồ F1-Score
        ax3.plot(epochs, self.history["train_f1"], "o-", label="Train F1", color="purple")
        ax3.plot(epochs, self.history["val_f1"], "s-", label="Val F1", color="teal")
        ax3.set_title("F1-Score (Weighted)", fontsize=13)
        ax3.set_xlabel("Epoch")
        ax3.set_ylabel("F1 Score")
        ax3.legend()
        ax3.grid(True, linestyle="--", alpha=0.6)
        plt.tight_layout()
        plt.savefig(save_path, dpi=300)
        plt.show()
        print(f"✔ Đã lưu biểu đồ training tại: {save_path}")


## 🚀 Dự đoán (Inference)
Hàm thực hiện dự đoán trên tập test và tạo file submission.

In [14]:
import zipfile
import os

def inference(model):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(device)

    # Tải từ điển
    vocab = Config.VOCAB
    ans_vocab = Config.ANS_VOCAB
    idx2ans = {v: k for k, v in ans_vocab.items()}

    # Chuẩn bị dữ liệu test
    test_ds = VQADataset(Path.TEST_JSON, Path.TEST_IMG_DIR, 
                         vocab, ans_vocab, train=False, test=True)
    loader = DataLoader(test_ds, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=2)

    # Tải model đã train
    if os.path.exists(Path.SAVE_PATH):
        print(f"Loading best checkpoint from {Path.SAVE_PATH}...")
        checkpoint = torch.load(Path.SAVE_PATH, map_location=device, weights_only=False)
        if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
            model.load_state_dict(checkpoint["model_state_dict"])
        else:
            model.load_state_dict(checkpoint)
    model.to(device)       
    model.eval()
    results = []
    print("Đang chạy inference...")
    with torch.no_grad():
        for img, q, qid in tqdm(loader, desc="Inference"):
            img, q = img.to(device), q.to(device)
            out = model(img, q)
            pred = out.argmax(1).cpu().tolist()

            for i in range(len(qid)):
                results.append([qid[i], idx2ans[pred[i]]])

    # Lưu CSV
    with open(Path.SUBMISSION_PATH, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["qid", "answer"])
        writer.writerows(results)

    print(f"Đã lưu kết quả tại: {Path.SUBMISSION_PATH}")

    # ---- ZIP submission ----
    zip_path = "submission.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(Path.SUBMISSION_PATH, arcname=os.path.basename(Path.SUBMISSION_PATH))

    print(f"Đã tạo file zip: {zip_path}")

## 🚦 Chạy quy trình
Thực thi việc huấn luyện và dự đoán.

In [ ]:
if __name__ == "__main__":
    # Bước 1: Khởi tạo model và data
    dataloader = Data_Loader()
    train_loader = dataloader.load_data(Path.TRAIN_JSON, Path.TRAIN_IMG_DIR, train=True)
    val_loader = dataloader.load_data(Path.VAL_JSON, Path.TRAIN_IMG_DIR, train=False, val=True)
    model = Model_ViT_Cross_attn(len(Config.VOCAB), len(Config.ANS_VOCAB))
    trainer = Tranning(model, train_loader, val_loader)
    trainer.train()
    trainer.plot_history()
    
    # # Bước 2: Dự đoán
    inference(model)

cuda
Loading best checkpoint from /kaggle/working/best_model.pth...
Đang chạy inference...


Inference: 100%|██████████| 3561/3561 [41:41<00:00,  1.42it/s]


Đã lưu kết quả tại: /kaggle/working/submission.csv
Đã tạo file zip: submission.zip
